# KNN-Enhanced BM25 Information Retrieval Model

This notebook implements a hybrid information retrieval system that combines:
- **BM25 (Okapi)** for initial document ranking
- **K-Nearest Neighbors (KNN)** for enhanced retrieval based on document similarity
- **Evaluation** on the Cranfield dataset using standard IR metrics

## Methodology
1. Load and preprocess the Cranfield dataset
2. Implement BM25 scoring for initial retrieval
3. Extract document features and apply KNN for similarity-based re-ranking
4. Evaluate performance using precision, recall, and MAP metrics


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import math
import warnings
warnings.filterwarnings('ignore')

# Text processing libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

# Machine learning libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Information retrieval libraries
import ir_datasets
try:
    from rank_bm25 import BM25Okapi
    print("✓ Using rank_bm25 library")
except ImportError:
    print("⚠ rank_bm25 not available, will implement BM25 manually")
    BM25Okapi = None

# Download required NLTK data
required_nltk_data = ['punkt', 'stopwords']
for data in required_nltk_data:
    try:
        nltk.data.find(f'tokenizers/{data}' if data == 'punkt' else f'corpora/{data}')
    except LookupError:
        print(f"Downloading {data}...")
        nltk.download(data)

print("✓ All libraries imported successfully!")
print("✓ NLTK data downloaded!")


In [ ]:
# Load Cranfield dataset
print("Loading Cranfield dataset...")
dataset = ir_datasets.load("cranfield")

# Extract documents, queries, and relevance judgments
documents = []
doc_ids = []
for doc in dataset.docs_iter():
    documents.append(doc.text)
    doc_ids.append(doc.doc_id)

queries = []
query_ids = []
for query in dataset.queries_iter():
    queries.append(query.text)
    query_ids.append(query.query_id)

# Store relevance judgments
qrels = defaultdict(set)
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0:  # Consider only relevant documents
        qrels[qrel.query_id].add(qrel.doc_id)

print(f"✓ Loaded {len(documents)} documents")
print(f"✓ Loaded {len(queries)} queries")
print(f"✓ Loaded relevance judgments for {len(qrels)} queries")

# Display sample data
print("\n--- Sample Document ---")
print(f"ID: {doc_ids[0]}")
print(f"Text: {documents[0][:200]}...")

print("\n--- Sample Query ---")
print(f"ID: {query_ids[0]}")
print(f"Text: {queries[0]}")

print(f"\n--- Sample Relevant Docs for Query {query_ids[0]} ---")
print(f"Relevant docs: {list(qrels[query_ids[0]])[:5]}...")


In [ ]:
# Text preprocessing function
def preprocess_text(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters and numbers
    3. Tokenizing
    4. Removing stopwords
    5. Stemming
    """
    # Initialize stemmer and stopwords
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    
    # Convert to lowercase and remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and stem
    processed_tokens = [
        stemmer.stem(token) 
        for token in tokens 
        if token not in stop_words and len(token) > 2
    ]
    
    return processed_tokens

# Preprocess all documents and queries
print("Preprocessing documents and queries...")
processed_documents = [preprocess_text(doc) for doc in documents]
processed_queries = [preprocess_text(query) for query in queries]

print(f"✓ Preprocessed {len(processed_documents)} documents")
print(f"✓ Preprocessed {len(processed_queries)} queries")

# Display preprocessing example
print("\n--- Preprocessing Example ---")
print(f"Original: {documents[0][:100]}...")
print(f"Processed: {' '.join(processed_documents[0][:20])}...")


In [ ]:
# BM25 Implementation (manual implementation if library not available)
class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(doc) for doc in corpus) / self.corpus_size
        self.corpus = corpus
        self.k1 = k1
        self.b = b
        
        # Calculate document frequencies
        self.doc_freqs = []
        self.idf = {}
        
        # Count term frequencies in each document
        for doc in corpus:
            frequencies = {}
            for word in doc:
                frequencies[word] = frequencies.get(word, 0) + 1
            self.doc_freqs.append(frequencies)
        
        # Calculate IDF values
        self.idf = self._calculate_idf()
    
    def _calculate_idf(self):
        """Calculate IDF values for all terms"""
        idf = {}
        all_words = set()
        for doc in self.corpus:
            all_words.update(doc)
        
        for word in all_words:
            containing_docs = sum(1 for doc in self.corpus if word in doc)
            idf[word] = math.log(self.corpus_size / containing_docs)
        
        return idf
    
    def get_scores(self, query):
        """Calculate BM25 scores for a query against all documents"""
        scores = np.zeros(self.corpus_size)
        
        for i, doc in enumerate(self.corpus):
            doc_len = len(doc)
            for word in query:
                if word in doc and word in self.idf:
                    freq = self.doc_freqs[i].get(word, 0)
                    score = self.idf[word] * freq * (self.k1 + 1) / (
                        freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    )
                    scores[i] += score
        
        return scores

# Initialize BM25
if BM25Okapi is not None:
    print("Using BM25Okapi from rank_bm25 library")
    bm25 = BM25Okapi(processed_documents)
else:
    print("Using manual BM25 implementation")
    bm25 = BM25(processed_documents)

print("✓ BM25 model initialized successfully!")


In [ ]:
# Extract TF-IDF features for KNN
print("Extracting TF-IDF features for documents...")

# Convert processed documents back to strings for TfidfVectorizer
doc_strings = [' '.join(doc) for doc in processed_documents]
query_strings = [' '.join(query) for query in processed_queries]

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Limit vocabulary size
    min_df=2,          # Ignore terms that appear in fewer than 2 documents
    max_df=0.8,        # Ignore terms that appear in more than 80% of documents
    ngram_range=(1, 2)  # Use unigrams and bigrams
)

# Fit and transform documents
doc_tfidf_matrix = tfidf_vectorizer.fit_transform(doc_strings)
print(f"✓ TF-IDF matrix shape: {doc_tfidf_matrix.shape}")

# Apply dimensionality reduction using SVD for better KNN performance
print("Applying SVD for dimensionality reduction...")
svd = TruncatedSVD(n_components=300, random_state=42)
doc_features_reduced = svd.fit_transform(doc_tfidf_matrix)

print(f"✓ Reduced feature matrix shape: {doc_features_reduced.shape}")
print(f"✓ Explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")

# Initialize KNN model
print("Initializing KNN model...")
knn_model = NearestNeighbors(
    n_neighbors=50,  # Number of neighbors to consider
    metric='cosine',  # Use cosine similarity
    algorithm='brute'  # Use brute force for cosine similarity
)

# Fit KNN model on document features
knn_model.fit(doc_features_reduced)
print("✓ KNN model trained successfully!")


In [ ]:
class HybridBM25KNN:
    def __init__(self, bm25_model, knn_model, tfidf_vectorizer, svd, doc_features, 
                 alpha=0.7, top_k_bm25=100, top_k_knn=50):
        """
        Hybrid retrieval system combining BM25 and KNN
        
        Args:
            bm25_model: Trained BM25 model
            knn_model: Trained KNN model  
            tfidf_vectorizer: Fitted TF-IDF vectorizer
            svd: Fitted SVD transformer
            doc_features: Document feature matrix
            alpha: Weight for BM25 scores (1-alpha for KNN)
            top_k_bm25: Number of top documents to consider from BM25
            top_k_knn: Number of neighbors to consider in KNN
        """
        self.bm25_model = bm25_model
        self.knn_model = knn_model
        self.tfidf_vectorizer = tfidf_vectorizer
        self.svd = svd
        self.doc_features = doc_features
        self.alpha = alpha
        self.top_k_bm25 = top_k_bm25
        self.top_k_knn = top_k_knn
    
    def search(self, query, top_k=10):
        """
        Perform hybrid search combining BM25 and KNN
        
        Args:
            query: Preprocessed query (list of tokens)
            top_k: Number of top documents to return
            
        Returns:
            List of (doc_index, combined_score) tuples
        """
        # Step 1: Get BM25 scores
        if BM25Okapi is not None:
            bm25_scores = self.bm25_model.get_scores(query)
        else:
            bm25_scores = self.bm25_model.get_scores(query)
        
        # Get top documents from BM25
        top_bm25_indices = np.argsort(bm25_scores)[::-1][:self.top_k_bm25]
        
        # Step 2: Transform query to feature space
        query_string = ' '.join(query)
        query_tfidf = self.tfidf_vectorizer.transform([query_string])
        query_features = self.svd.transform(query_tfidf)
        
        # Step 3: Find KNN neighbors based on query
        knn_distances, knn_indices = self.knn_model.kneighbors(query_features, 
                                                              n_neighbors=self.top_k_knn)
        
        # Convert distances to similarities (for cosine distance)
        knn_similarities = 1 - knn_distances[0]
        
        # Step 4: Combine scores
        combined_scores = {}
        
        # Normalize BM25 scores
        max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
        normalized_bm25 = bm25_scores / max_bm25
        
        # Add BM25 scores for top documents
        for idx in top_bm25_indices:
            combined_scores[idx] = self.alpha * normalized_bm25[idx]
        
        # Add KNN similarities
        max_knn = np.max(knn_similarities) if np.max(knn_similarities) > 0 else 1
        for i, doc_idx in enumerate(knn_indices[0]):
            knn_score = (1 - self.alpha) * (knn_similarities[i] / max_knn)
            if doc_idx in combined_scores:
                combined_scores[doc_idx] += knn_score
            else:
                combined_scores[doc_idx] = knn_score
        
        # Sort by combined score and return top k
        sorted_results = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results[:top_k]

# Initialize hybrid model
hybrid_model = HybridBM25KNN(
    bm25_model=bm25,
    knn_model=knn_model,
    tfidf_vectorizer=tfidf_vectorizer,
    svd=svd,
    doc_features=doc_features_reduced,
    alpha=0.7,  # 70% BM25, 30% KNN
    top_k_bm25=100,
    top_k_knn=50
)

print("✓ Hybrid BM25+KNN model initialized successfully!")


In [ ]:
def calculate_precision_recall(retrieved_docs, relevant_docs, k=10):
    """
    Calculate precision and recall at k
    
    Args:
        retrieved_docs: List of retrieved document IDs
        relevant_docs: Set of relevant document IDs
        k: Number of top documents to consider
        
    Returns:
        precision: Precision at k
        recall: Recall at k
    """
    retrieved_at_k = set(retrieved_docs[:k])
    relevant_retrieved = retrieved_at_k.intersection(relevant_docs)
    
    precision = len(relevant_retrieved) / len(retrieved_at_k) if len(retrieved_at_k) > 0 else 0
    recall = len(relevant_retrieved) / len(relevant_docs) if len(relevant_docs) > 0 else 0
    
    return precision, recall

def calculate_average_precision(retrieved_docs, relevant_docs):
    """
    Calculate Average Precision (AP) for a single query
    
    Args:
        retrieved_docs: List of retrieved document IDs in ranked order
        relevant_docs: Set of relevant document IDs
        
    Returns:
        Average precision score
    """
    if len(relevant_docs) == 0:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    
    return score / len(relevant_docs)

def calculate_map(all_retrieved, all_relevant):
    """
    Calculate Mean Average Precision (MAP) across all queries
    
    Args:
        all_retrieved: Dictionary {query_id: [retrieved_doc_ids]}
        all_relevant: Dictionary {query_id: set(relevant_doc_ids)}
        
    Returns:
        MAP score
    """
    ap_scores = []
    for query_id in all_retrieved:
        if query_id in all_relevant:
            ap = calculate_average_precision(all_retrieved[query_id], all_relevant[query_id])
            ap_scores.append(ap)
    
    return np.mean(ap_scores) if ap_scores else 0.0

def evaluate_model(model, queries, query_ids, qrels, doc_ids, top_k=10, model_name="Model"):
    """
    Evaluate a retrieval model on all queries
    
    Args:
        model: Retrieval model with search method
        queries: List of preprocessed queries
        query_ids: List of query IDs
        qrels: Dictionary of relevance judgments
        doc_ids: List of document IDs
        top_k: Number of top documents to retrieve
        model_name: Name of the model for reporting
        
    Returns:
        Dictionary with evaluation metrics
    """
    all_retrieved = {}
    all_relevant = {}
    
    precisions_at_k = []
    recalls_at_k = []
    
    print(f"Evaluating {model_name}...")
    
    for i, (query, query_id) in enumerate(zip(queries, query_ids)):
        if i % 50 == 0:
            print(f"Processing query {i+1}/{len(queries)}")
        
        # Get search results
        if hasattr(model, 'search'):
            # Hybrid model
            results = model.search(query, top_k=top_k)
            retrieved_doc_indices = [result[0] for result in results]
        else:
            # BM25 model
            if BM25Okapi is not None:
                scores = model.get_scores(query)
            else:
                scores = model.get_scores(query)
            retrieved_doc_indices = np.argsort(scores)[::-1][:top_k]
        
        # Convert indices to document IDs
        retrieved_doc_ids = [doc_ids[idx] for idx in retrieved_doc_indices]
        
        # Get relevant documents for this query
        relevant_doc_ids = qrels.get(query_id, set())
        
        # Store for MAP calculation
        all_retrieved[query_id] = retrieved_doc_ids
        all_relevant[query_id] = relevant_doc_ids
        
        # Calculate precision and recall at k
        precision, recall = calculate_precision_recall(retrieved_doc_ids, relevant_doc_ids, k=top_k)
        precisions_at_k.append(precision)
        recalls_at_k.append(recall)
    
    # Calculate overall metrics
    avg_precision = np.mean(precisions_at_k)
    avg_recall = np.mean(recalls_at_k)
    map_score = calculate_map(all_retrieved, all_relevant)
    
    results = {
        'model_name': model_name,
        'precision_at_k': avg_precision,
        'recall_at_k': avg_recall,
        'map': map_score,
        'num_queries': len(queries)
    }
    
    print(f"✓ {model_name} Evaluation Complete!")
    return results

print("✓ Evaluation functions defined!")


In [ ]:
# Evaluate both BM25 and Hybrid BM25+KNN models
print("Starting comprehensive evaluation...")
print("=" * 60)

# Evaluate BM25 baseline
bm25_results = evaluate_model(
    model=bm25,
    queries=processed_queries,
    query_ids=query_ids,
    qrels=qrels,
    doc_ids=doc_ids,
    top_k=10,
    model_name="BM25 Baseline"
)

print("\n" + "=" * 60)

# Evaluate Hybrid BM25+KNN model
hybrid_results = evaluate_model(
    model=hybrid_model,
    queries=processed_queries,
    query_ids=query_ids,
    qrels=qrels,
    doc_ids=doc_ids,
    top_k=10,
    model_name="Hybrid BM25+KNN"
)

print("\n" + "=" * 60)
print("EVALUATION RESULTS SUMMARY")
print("=" * 60)

# Create results comparison
results_df = pd.DataFrame([bm25_results, hybrid_results])
print(results_df.to_string(index=False, float_format='%.4f'))

print("\n" + "=" * 60)
print("PERFORMANCE IMPROVEMENT")
print("=" * 60)

# Calculate improvements
precision_improvement = ((hybrid_results['precision_at_k'] - bm25_results['precision_at_k']) / 
                        bm25_results['precision_at_k'] * 100) if bm25_results['precision_at_k'] > 0 else 0

recall_improvement = ((hybrid_results['recall_at_k'] - bm25_results['recall_at_k']) / 
                     bm25_results['recall_at_k'] * 100) if bm25_results['recall_at_k'] > 0 else 0

map_improvement = ((hybrid_results['map'] - bm25_results['map']) / 
                  bm25_results['map'] * 100) if bm25_results['map'] > 0 else 0

print(f"Precision@10 improvement: {precision_improvement:+.2f}%")
print(f"Recall@10 improvement: {recall_improvement:+.2f}%")
print(f"MAP improvement: {map_improvement:+.2f}%")

# Store results for plotting
evaluation_results = {
    'bm25': bm25_results,
    'hybrid': hybrid_results
}
